In [2]:
##################------------------cell1--------------------------------#############3



import pandas as pd
import numpy as np

# ---------------------------
# Load Dataset
# ---------------------------



df = pd.read_csv(
    "phase2_OctNov_combined_session_features.csv",
    parse_dates=["session_start", "session_end"]
)

print("="*60)
print("DATASET INFORMATION")
print("="*60)

print("Rows :", len(df))
print("Columns :", len(df.columns))
print("Unique Users :", df["user_id"].nunique())
print("Date Range :", df["session_start"].min(), "to", df["session_end"].max())

df.head()

DATASET INFORMATION
Rows : 24100416
Columns : 17
Unique Users : 5316561
Date Range : 2019-10-01 00:00:00+00:00 to 2019-11-30 23:59:59+00:00


,user_session,user_id,session_start,session_end,num_events,num_views,num_carts,num_remove_from_cart,num_unique_products,num_unique_categories,avg_price,max_price,min_price,num_brands,session_duration,hour_of_day,made_purchase
0,00000042-3e3f-42f9-810d-f3d264139c50,515483062,2019-10-18 10:54:45+00:00,2019-10-18 10:55:20+00:00,2,2,0,0,1,1,64.350000,64.35,64.35,1,35.0,10,0
1,00000056-a206-40dd-b174-a072550fa38c,513782162,2019-10-31 06:23:12+00:00,2019-10-31 06:31:25+00:00,8,8,0,0,7,2,715.023750,1349.46,29.60,3,493.0,6,0
2,00000083-8816-4d58-a9b8-f52f54186edc,546521725,2019-10-06 11:24:45+00:00,2019-10-06 11:34:30+00:00,18,16,1,0,14,1,584.765556,1747.79,152.58,3,585.0,11,1
3,000001fd-1f89-45e8-a3ce-fe3218cabfad,560486342,2019-10-25 08:30:42+00:00,2019-10-25 08:39:11+00:00,11,6,4,0,3,1,164.438182,171.90,131.51,1,509.0,8,1
4,000003eb-b63e-45d9-9f26-f229057c654a,512483064,2019-10-03 11:28:52+00:00,2019-10-03 11:28:52+00:00,1,1,0,0,1,1,195.420000,195.42,195.42,1,0.0,11,0


In [3]:
########## ----cell 2--------------------- ####################33

# --------------------------------------------------
# Feature Engineering
# --------------------------------------------------

reference_date = df["session_end"].max() + pd.Timedelta(days=1)

# Better monetary approximation
df["estimated_session_value"] = np.where(
    df["made_purchase"] == 1,
    df["avg_price"] * df["num_unique_products"],
    0
)

user_features = df.groupby("user_id").agg(

    total_sessions=("user_session", "count"),

    purchase_sessions=("made_purchase", "sum"),

    total_events=("num_events", "sum"),

    total_views=("num_views", "sum"),

    avg_session_duration=("session_duration", "mean"),

    total_carts=("num_carts", "sum"),

    total_removes=("num_remove_from_cart", "sum"),

    avg_unique_products=("num_unique_products", "mean"),

    avg_unique_categories=("num_unique_categories", "mean"),

    avg_brands=("num_brands", "mean"),

    avg_hour_of_day=("hour_of_day", "mean"),

    estimated_total_value=("estimated_session_value", "sum"),

    estimated_avg_value=("estimated_session_value", "mean"),

    last_session=("session_start", "max")

).reset_index()

print("Aggregation Complete.")
print("Users:", len(user_features))

Aggregation Complete.
Users: 5316561


In [6]:
# --------------------------------------------------
# Derived Features
# --------------------------------------------------

user_features["recency_days"] = (
    reference_date - user_features["last_session"]
).dt.days

user_features["purchase_rate"] = (
    user_features["purchase_sessions"] /
    user_features["total_sessions"]
)

user_features["net_cart_activity"] = (
    user_features["total_carts"] -
    user_features["total_removes"]
)

user_features.drop(columns=["last_session"], inplace=True)

print("Derived Features Created.")

user_features.head()

Derived Features Created.


,user_id,total_sessions,purchase_sessions,total_events,total_views,avg_session_duration,total_carts,total_removes,avg_unique_products,avg_unique_categories,avg_brands,avg_hour_of_day,estimated_total_value,estimated_avg_value,recency_days,purchase_rate,net_cart_activity
0,10300217,1,0,1,1,0.000000,0,0,1.000000,1.0,1.000000,6.000000,0.0,0.0,25,0.0,0
1,29515875,7,0,11,11,15.857143,0,0,1.142857,1.0,1.142857,5.142857,0.0,0.0,11,0.0,0
2,31198833,5,0,20,20,372.200000,0,0,3.800000,1.0,1.600000,1.800000,0.0,0.0,12,0.0,0
3,33869381,1,0,1,1,0.000000,0,0,1.000000,1.0,1.000000,20.000000,0.0,0.0,39,0.0,0
4,34916060,1,0,1,1,0.000000,0,0,1.000000,1.0,1.000000,7.000000,0.0,0.0,7,0.0,0


In [7]:
# --------------------------------------------------
# Validation
# --------------------------------------------------

user_features["estimated_total_value"] = (
    user_features["estimated_total_value"].fillna(0)
)

user_features["estimated_avg_value"] = (
    user_features["estimated_avg_value"].fillna(0)
)

print("="*60)
print("Missing Values")
print("="*60)

print(user_features.isnull().sum())

print("\nSummary Statistics")

display(user_features.describe())

print("\nSample Records")

display(user_features.head(10))

never_purchase = (user_features["purchase_sessions"] == 0).sum()

print(f"\nUsers with no purchases : {never_purchase:,}")
print(f"Percentage : {(never_purchase/len(user_features))*100:.2f}%")

Missing Values
user_id                  0
total_sessions           0
purchase_sessions        0
total_events             0
total_views              0
avg_session_duration     0
total_carts              0
total_removes            0
avg_unique_products      0
avg_unique_categories    0
avg_brands               0
avg_hour_of_day          0
estimated_total_value    0
estimated_avg_value      0
recency_days             0
purchase_rate            0
net_cart_activity        0
dtype: int64

Summary Statistics


,user_id,total_sessions,purchase_sessions,total_events,total_views,avg_session_duration,total_carts,total_removes,avg_unique_products,avg_unique_categories,avg_brands,avg_hour_of_day,estimated_total_value,estimated_avg_value,recency_days,purchase_rate,net_cart_activity
count,5.316561e+06,5.316561e+06,5.316561e+06,5.316561e+06,5.316561e+06,5.316561e+06,5.316561e+06,5316561.0,5.316561e+06,5.316561e+06,5.316561e+06,5.316561e+06,5.316561e+06,5.316561e+06,5.316561e+06,5.316561e+06,5.316561e+06
mean,5.472979e+08,4.533084e+00,2.793533e-01,2.068080e+01,1.962462e+01,7.298397e+02,7.439836e-01,0.0,2.427361e+00,1.256187e+00,1.616140e+00,1.080640e+01,2.101719e+02,3.366166e+01,2.248865e+01,4.797789e-02,7.439836e-01
std,2.267393e+07,1.659413e+01,1.327676e+00,5.391708e+01,5.206749e+01,1.658260e+04,3.085803e+00,0.0,2.681291e+00,6.406773e-01,1.096257e+00,4.332413e+00,1.502451e+03,2.128221e+02,1.726795e+01,1.678746e-01,3.085803e+00
min,1.030022e+07,1.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00
25%,5.260510e+08,1.000000e+00,0.000000e+00,2.000000e+00,2.000000e+00,0.000000e+00,0.000000e+00,0.0,1.000000e+00,1.000000e+00,1.000000e+00,8.000000e+00,0.000000e+00,0.000000e+00,8.000000e+00,0.000000e+00,0.000000e+00
50%,5.530729e+08,2.000000e+00,0.000000e+00,5.000000e+00,5.000000e+00,8.666667e+01,0.000000e+00,0.0,1.666667e+00,1.000000e+00,1.000000e+00,1.093750e+01,0.000000e+00,0.000000e+00,1.800000e+01,0.000000e+00,0.000000e+00
75%,5.663136e+08,5.000000e+00,0.000000e+00,1.800000e+01,1.700000e+01,2.542500e+02,0.000000e+00,0.0,3.000000e+00,1.250000e+00,2.000000e+00,1.400000e+01,0.000000e+00,0.000000e+00,3.600000e+01,0.000000e+00,0.000000e+00
max,5.799699e+08,2.255800e+04,3.420000e+02,2.292900e+04,2.292600e+04,2.618715e+06,7.200000e+02,0.0,1.376000e+03,7.100000e+01,1.790000e+02,2.300000e+01,3.802415e+05,3.624926e+04,6.100000e+01,1.000000e+00,7.200000e+02



Sample Records


,user_id,total_sessions,purchase_sessions,total_events,total_views,avg_session_duration,total_carts,total_removes,avg_unique_products,avg_unique_categories,avg_brands,avg_hour_of_day,estimated_total_value,estimated_avg_value,recency_days,purchase_rate,net_cart_activity
0,10300217,1,0,1,1,0.000000,0,0,1.000000,1.0,1.000000,6.000000,0.0,0.0,25,0.0,0
1,29515875,7,0,11,11,15.857143,0,0,1.142857,1.0,1.142857,5.142857,0.0,0.0,11,0.0,0
2,31198833,5,0,20,20,372.200000,0,0,3.800000,1.0,1.600000,1.800000,0.0,0.0,12,0.0,0
3,33869381,1,0,1,1,0.000000,0,0,1.000000,1.0,1.000000,20.000000,0.0,0.0,39,0.0,0
4,34916060,1,0,1,1,0.000000,0,0,1.000000,1.0,1.000000,7.000000,0.0,0.0,7,0.0,0
5,41798457,1,0,1,1,0.000000,0,0,1.000000,1.0,1.000000,8.000000,0.0,0.0,5,0.0,0
6,42896738,1,0,3,3,221.000000,0,0,3.000000,1.0,3.000000,11.000000,0.0,0.0,19,0.0,0
7,43295513,1,0,1,1,0.000000,0,0,1.000000,1.0,1.000000,11.000000,0.0,0.0,5,0.0,0
8,44378150,1,0,1,1,0.000000,0,0,1.000000,1.0,1.000000,14.000000,0.0,0.0,11,0.0,0
9,49484535,18,0,18,18,0.000000,0,0,1.000000,1.0,1.000000,13.944444,0.0,0.0,1,0.0,0



Users with no purchases : 4,625,758
Percentage : 87.01%


In [8]:
# --------------------------------------------------
# Save Output
# --------------------------------------------------

output_path = "phase5_user_rfm_features.csv"

user_features.to_csv(
    output_path,
    index=False
)

print("="*60)
print("PROCESS COMPLETED SUCCESSFULLY")
print("="*60)

print("Saved To:")
print(output_path)

print("\nFinal Dataset Shape:", user_features.shape)
import os

print("Current Working Directory:")
print(os.getcwd())

PROCESS COMPLETED SUCCESSFULLY
Saved To:
phase5_user_rfm_features.csv

Final Dataset Shape: (5316561, 17)
Current Working Directory:
c:\Users\sajal\Downloads\Retail-Demand-Sustainability-Project--main


In [9]:
# ==========================================================
# DESCRIPTIVE STATISTICS & FEATURE DISTRIBUTION
# ==========================================================

print("="*80)
print("DESCRIPTIVE STATISTICS OF CUSTOMER FEATURES")
print("="*80)

summary = user_features.describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).T

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.float_format", "{:.2f}".format)

print(summary)

print("\n" + "="*80)
print("FEATURE INTERPRETATION")
print("="*80)

print(f"""
Number of Customers          : {len(user_features):,}

Features Analysed:
1. Recency (recency_days)
2. Frequency (total_sessions)
3. Purchase Behaviour (purchase_sessions, purchase_rate)
4. Monetary Value (estimated_total_value, estimated_avg_value)
5. Engagement (avg_session_duration)
6. Cart Activity (total_carts, total_removes, net_cart_activity)
7. Browsing Behaviour (total_views, total_events,
   avg_unique_products, avg_unique_categories, avg_brands)

Percentiles shown:
25%  -> First Quartile
50%  -> Median
75%  -> Third Quartile
90%  -> Top 10% Customers
95%  -> Top 5% Customers
99%  -> Top 1% Customers
""")

print("="*80)
print("CUSTOMERS WITH NO PURCHASE")
print("="*80)

never_purchase = (user_features["purchase_sessions"] == 0).sum()
buyers = len(user_features) - never_purchase

print(f"Customers who never purchased : {never_purchase:,}")
print(f"Customers who purchased       : {buyers:,}")
print(f"Non-buyers (%)                : {100*never_purchase/len(user_features):.2f}%")
print(f"Buyers (%)                    : {100*buyers/len(user_features):.2f}%")

print("\n" + "="*80)
print("POTENTIAL SKEWNESS CHECK")
print("="*80)

skew_features = [
    "total_sessions",
    "total_events",
    "total_views",
    "estimated_total_value",
    "total_carts"
]

for col in skew_features:

    median = user_features[col].median()
    p99 = user_features[col].quantile(0.99)

    print(f"{col:25s} Median = {median:10.2f} | 99th Percentile = {p99:10.2f}")

DESCRIPTIVE STATISTICS OF CUSTOMER FEATURES
                           count         mean         std         min  \
user_id               5316561.00 547297890.52 22673930.11 10300217.00   
total_sessions        5316561.00         4.53       16.59        1.00   
purchase_sessions     5316561.00         0.28        1.33        0.00   
total_events          5316561.00        20.68       53.92        1.00   
total_views           5316561.00        19.62       52.07        0.00   
avg_session_duration  5316561.00       729.84    16582.60        0.00   
total_carts           5316561.00         0.74        3.09        0.00   
total_removes         5316561.00         0.00        0.00        0.00   
avg_unique_products   5316561.00         2.43        2.68        1.00   
avg_unique_categories 5316561.00         1.26        0.64        1.00   
avg_brands            5316561.00         1.62        1.10        1.00   
avg_hour_of_day       5316561.00        10.81        4.33        0.00   
estimat